In [7]:
import pyodbc
import pandas as pd
from collections import Counter

# 1. Establish connection to SQL Server
conn = pyodbc.connect('Driver={SQL Server};'
                      'Server=.;'
                      'Database=PizzaSalesDB;'
                      'Trusted_Connection=yes;')
cursor = conn.cursor()

# 2. Extract data from the main table
print("Extracting data from Pizza_Sales_Data...")
query = "SELECT order_id, pizza_ingredients FROM [Pizza_Sales_Data]"
df = pd.read_sql(query, conn)

# 3. Process data to create the Bridge Table (Fact_Order_Ingredients)
print("Processing ingredients for all orders...")
bridge_data = []
for index, row in df.iterrows():
    order_id = row['order_id']
    # Split ingredients by comma and remove extra spaces
    ingredients = [i.strip() for i in row['pizza_ingredients'].split(',')]
    for ing in ingredients:
        bridge_data.append({'order_id': order_id, 'ingredient_name': ing})

df_bridge = pd.DataFrame(bridge_data)

# 4. Process data to create the Dimension Table (Dim_Ingredients)
all_ingredients_list = [item['ingredient_name'] for item in bridge_data]
ingredients_count = Counter(all_ingredients_list)
ingredients_df = pd.DataFrame(ingredients_count.items(), columns=['Ingredient', 'Count'])

# 5. Function to safely upload DataFrames to SQL Server using fast_executemany
def safe_upload(dataframe, table_name):
    print(f"Uploading {table_name}...")
    # Drop table if it already exists to avoid conflicts
    cursor.execute(f"IF OBJECT_ID('{table_name}', 'U') IS NOT NULL DROP TABLE {table_name}")
    
    # Create table with explicit data types to avoid precision errors
    if table_name == 'Dim_Ingredients':
        cursor.execute(f"CREATE TABLE {table_name} (Ingredient NVARCHAR(255), Count INT)")
        insert_query = f"INSERT INTO {table_name} (Ingredient, Count) VALUES (?, ?)"
    else: 
        # For Fact_Order_Ingredients
        cursor.execute(f"CREATE TABLE {table_name} (order_id INT, ingredient_name NVARCHAR(255))")
        insert_query = f"INSERT INTO {table_name} (order_id, ingredient_name) VALUES (?, ?)"
    
    # Enable fast execution for high performance
    cursor.fast_executemany = True
    
    # Convert DataFrame to list of tuples for executemany
    values_to_insert = [tuple(x) for x in dataframe.values]
    
    # Execute insert and commit
    cursor.executemany(insert_query, values_to_insert)
    conn.commit()
    print(f"✅ Successfully uploaded {len(dataframe)} rows to {table_name}")

# 6. Upload both tables
safe_upload(ingredients_df, 'Dim_Ingredients')
safe_upload(df_bridge, 'Fact_Order_Ingredients')

# 7. Close connections
cursor.close()
conn.close()
print("🎉 All tasks completed successfully! Ready for Power BI.")

Extracting data from Pizza_Sales_Data...
Processing ingredients for all orders...


C:\Users\tarek\AppData\Local\Temp\ipykernel_5764\2306253955.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Uploading Dim_Ingredients...
✅ Successfully uploaded 65 rows to Dim_Ingredients
Uploading Fact_Order_Ingredients...
✅ Successfully uploaded 267576 rows to Fact_Order_Ingredients
🎉 All tasks completed successfully! Ready for Power BI.
